# CASM voltage dumps: trigger, read, correlate

Runs on **casm-corr1**; kernel = the offline venv
(`/home/casm/software/dev/casm_venvs/casm_offline_env`).

Voltages are 4+4-bit complex, one byte per sample: 3072 channels over
390.6-484.4 MHz in six 512-channel streams, stream 0 at the top of the band.
Streams 0-2 are written on casm-corr1, streams 3-5 on casm-corr2. Native
sampling is 32.768 us; one second of the full array is ~12.4 GB on disk.

## 1. Dump and gather

`dump_voltages` triggers the daemons on both nodes, waits for the files,
pulls the corr2 streams into your directory (corr1 ones are symlinked), and
returns what `VoltageReader` needs. The ring buffer holds ~26 s of the past,
so `seconds=` is capped there; an explicit UTC window can also be in the
near future. Check the plan and disk space first with `dry_run=True`.

In [ ]:
from casm_t2.dump_client import dump_voltages

out_dir = "/home/casm/software/vishnu/VOLTAGE_TESTS"
seconds = 2
utc_start = "2026-07-30-22:25:22.958"   # PSRDADA UTC
utc_stop = "2026-07-30-22:25:24.958"

# stream 0: 468.750-484.375 MHz (corr1)    stream 3: 421.875-437.500 MHz (corr2)
# stream 1: 453.125-468.750 MHz (corr1)    stream 4: 406.250-421.875 MHz (corr2)
# stream 2: 437.500-453.125 MHz (corr1)    stream 5: 390.625-406.250 MHz (corr2)
subband_streams = [1, 2]   # 437.5-468.75 MHz, covers the 440-465 MHz live band

dump_voltages(out_dir, seconds=seconds, dry_run=True)

In [ ]:
data_dir, prefix = dump_voltages(out_dir, seconds=seconds)

# other forms:
# data_dir, prefix = dump_voltages(out_dir, start=utc_start, stop=utc_stop)
# data_dir, prefix = dump_voltages(out_dir, seconds=seconds, streams=subband_streams)

data_dir, prefix

## 2. Read

Reads take `seconds=`/`offset_seconds=` (or exact samples via `n_time=`/
`time_offset=`), `subbands=` for a contiguous stream selection, `snaps=`
for a subset of SNAP boards. Default return is the raw SNAP layout,
`{snap: (n_time, n_chan, 12)}` complex64 with values -8..7; pass
`antenna_csv=` for one `(n_time, n_chan, n_ant)` array in CSV row order.

In [ ]:
from casm_io.voltage import VoltageReader

reader = VoltageReader(data_dir, prefix)
print("streams on disk:", reader.subbands_found)

res = reader.read_full_band(seconds=0.1, subbands=[1, 2], snaps=[0, 3],
                            verbose=False)
v0 = res.voltages[0]
print(f"SNAP 0: {v0.shape} {v0.dtype}   (time, channel, adc)")
print(f"freq: {res.freq_mhz[0]:.4f} -> {res.freq_mhz[-1]:.4f} MHz")

## 3. Long reads: gulp iteration

Unpacked complex64 is ~8x the disk size, so long reads go in chunks.
`iter_full_band` yields RAM-sized pieces; accumulate across them — here,
per-SNAP power spectra.

In [ ]:
import numpy as np

spec = {}          # snap -> (n_chan, n_adc) accumulated |v|^2
n = 0
for chunk in reader.iter_full_band(seconds=2, snaps=[0, 3]):
    for s, v in chunk.voltages.items():
        spec[s] = spec.get(s, 0) + np.sum(v.real**2 + v.imag**2, axis=0)
    n += next(iter(chunk.voltages.values())).shape[0]

freq = chunk.freq_mhz
header = chunk.header
print(f"accumulated {n} samples ({n * 32.768e-6:.2f} s) for SNAPs {sorted(spec)}")

## 4. Visibilities (optional)

`correlate` multiplies the inputs pairwise, then averages. `tint_s=None`
keeps the native 32.768 us resolution; anything else rounds to whole
samples. A snap dict is stacked in (snap, adc) order — the labels are in
`.inputs`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from casm_io.voltage import correlate

small = reader.read_full_band(seconds=0.005, subbands=[1], snaps=[0, 3],
                              verbose=False)

native = correlate(small.voltages)              # native resolution
avg = correlate(small.voltages, tint_s=0.001)   # 1 ms integrations

print("native:", native.vis.shape, "(bin, channel, input, input)")
print("1 ms:  ", avg.vis.shape, f"tint = {avg.tint_samples} samples")
print("input 0 is (snap, adc) =", avg.inputs[0])

## 5. Autocorrelation bandpasses

One figure per SNAP, every ADC labelled, local time range in the title.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from casm_io.autocorr import psrdada_to_unix
from casm_io._time import format_time_span

t0 = psrdada_to_unix(header.get("DUMP_UTC_START", header["UTC_START"]))
when = format_time_span(t0, t0 + n * 32.768e-6, "America/Los_Angeles")

for snap in sorted(spec):
    fig, axes = plt.subplots(3, 4, figsize=(12, 7), sharex=True)
    for adc in range(spec[snap].shape[1]):
        ax = axes.flat[adc]
        ax.semilogy(freq, spec[snap][:, adc] / n, lw=0.6)
        ax.set_title(f"ADC {adc}", fontsize=9)
    for ax in axes[-1]:
        ax.set_xlabel("Frequency (MHz)")
    for ax in axes[:, 0]:
        ax.set_ylabel("mean |v|$^2$")
    fig.suptitle(f"SNAP {snap} autocorrelations — {when}")
    fig.tight_layout()

## 6. Cleaning up

Nothing deletes dumps automatically (the T3 janitor is off). Remove your
working directory when done, and the originals under
`/mnt/nvme4/data/casm/cand_dumps/stream_N/` on both nodes if nobody needs
them. The same flow scripted: `examples/voltage_dumps.py`; comparison
against the correlator: `casm-autocorr`.